## Задание 5

Напишите функцию `get_normalize`, которая будет принимать тензор матрицу с признаками объектов из какого-то датасета с картинками и будет возвращать поканальное среднее и поканальное стандартное отклонение. Гарантируется, что матрица будет иметь размер `[N, C, H, W]`, где `N` это количество объектов, `C` - количество каналов, `H`, `W` - размеры изображений. Нужно вернуть кортеж из двух тензоров длины `C`.

Ваша функция должна иметь следующую сигнатуру `def get_normalize(features: torch.Tensor):`



In [ ]:
import torch


def get_normalize(features: torch.Tensor):
    means = features.mean(dim=(0, 2, 3))
    stds = features.std(dim=(0, 2, 3))

    return means, stds


## Задание 6
Напишите функцию `get_augmentations`, которая будет возвращать готовые аугментации для обучающей выборки и для тестовой выборки. Она должна иметь следующую сигнатуру: `def get_augmentations(train: bool = True) -> T.Compose:`.

Примените следующие аугментации:

 1. Измените размер изображения, чтобы его размер был 224 на 224 пикселя (и для обучающей и для тестовой выборки)
 2. Примените какие-нибудь аугментации из тех, что мы изучили на занятии (только для обучающей)
 3. Преобразуйте картинку в тензор
 4. Примените нормализацию для датасета CIFAR10

Понятно, что изменение размера к 224 на 224 и нормализация для CIFAR10 вместе сочетаются плохо. Поэтому в дальнейшем, когда будете использовать эту функцию для получения аугментаций для вашего нового датасета, не забудьте поменять их на специфичные ему.



In [ ]:
import torchvision.transforms as T


def get_augmentations(train: bool = True) -> T.Compose:
    if train:
        return T.Compose(
            [
                T.Resize((224, 224)),
                T.RandomHorizontalFlip(),
                T.ToTensor(),
                T.Normalize(mean=(0.49139968, 0.48215841, 0.44653091), std=(0.24703223, 0.24348513, 0.26158784))
            ]
        )
    return T.Compose(
        [
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=(0.49139968, 0.48215841, 0.44653091), std=(0.24703223, 0.24348513, 0.26158784))
        ]
    )


## Задание 7

Напишите функцию `predict`. Она должна принимать на вход нейронную сеть, даталоадер и `torch.device`. Она должна иметь следующую сигнатуру: `def predict(model: nn.Module, loader: DataLoader, device: torch.device):`

Внутри функции сделайте следующие шаги:

 1. Создайте пустой список для хранения предсказаний
 2. Проитерируйтесь по даталоадеру
 3. На каждой итерации
        - сделайте forward pass для батча
        - посчитайте классы как аргмакс по выходу нейросети, по логитам
        - добавьте тензор с предсказаниями в список
 4. Сделайте конкатенацию всех предсказаний и верните этот тензор длины `N`, по числу объектов в датасете

Ваша функция должна возвращать тензор с **классами**.



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


@torch.inference_mode()
def predict(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()

    prediction = []

    for x, _ in loader:
        output = model(x.to(device)).cpu()

        prediction.append(torch.argmax(output, dim=1))

    prediction = torch.cat(prediction)

    return prediction


## Задание 8

Обучите модель из двух сверточных слоев на датасете CIFAR10, добейтесь значения метрики Accuracy в 70% на тестовой выборке. Ограничения касаются только количества сверточных слоев в архитектуре модели, можно использовать любые клевые штуки, что мы прошли на занятии. Вам нужно сдать код с функцией, которая возвращает модель, назовите эту функцию `create_simple_conv_cifar`. Она не принимает аргументов и возвращает модель. Также сдайте предсказание для тестовой выборки датасета CIFAR10, воспользуйтесь функцией `predict` из предыдущего задания. Воспользуйтесь `torch.save` для записи тензора с результатом предсказания на диск.


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from IPython.display import clear_output
from torch.optim import Adam
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from tqdm import tqdm


def set_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)


def train(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: Optimizer,
    loss_fn,
    device: torch.device,
) -> tuple[float, float]:
    model.train()

    total_loss = 0
    total = 0
    correct = 0

    for x, y in tqdm(data_loader, desc="Train"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        output = model(x)

        loss = loss_fn(output, y)

        loss.backward()

        total_loss += loss.item()

        optimizer.step()

        _, y_pred = torch.max(output, 1)
        total += y.size(0)
        correct += (y_pred == y).sum().item()

    return total_loss / len(data_loader), correct / total


@torch.inference_mode()
def evaluate(
    model: nn.Module, data_loader: DataLoader, loss_fn, device: torch.device
) -> tuple[float, float]:
    model.eval()

    total_loss = 0
    total = 0
    correct = 0

    for x, y in tqdm(data_loader, desc="Evaluate"):
        x, y = x.to(device), y.to(device)

        output = model(x)

        loss = loss_fn(output, y)

        total_loss += loss.item()

        _, y_pred = torch.max(output, 1)
        total += y.size(0)
        correct += (y_pred == y).sum().item()

    return total_loss / len(data_loader), correct / total


def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    train_accuracy: list[float],
    valid_accuracy: list[float],
    title: str,
):
    plt.figure(figsize=(16, 8))

    plt.title(title + " loss")

    plt.plot(train_loss, label="Train loss")
    plt.plot(valid_loss, label="Valid loss")
    plt.legend()
    plt.grid()

    plt.show()

    plt.figure(figsize=(16, 8))

    plt.title(title + " accuracy")

    plt.plot(train_accuracy, label="Train accuracy")
    plt.plot(valid_accuracy, label="Valid accuracy")
    plt.legend()
    plt.grid()

    plt.show()


def whole_train_valid_cycle(
    model, train_loader, valid_loader, optimizer, loss_fn, device, threshold, title
):
    train_loss_history, valid_loss_history = [], []
    train_accuracy_history, valid_accuracy_history = [], []

    for epoch in range(100):
        train_loss, train_accuracy = train(
            model, train_loader, optimizer, loss_fn, device
        )
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn, device)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        train_accuracy_history.append(train_accuracy)
        valid_accuracy_history.append(valid_accuracy)

        clear_output(wait=True)

        plot_stats(
            train_loss_history,
            valid_loss_history,
            train_accuracy_history,
            valid_accuracy_history,
            title,
        )

        if valid_accuracy >= threshold:
            break


def main(func, threshold, title):
    set_seed(0xDEADF00D)

    dataset = CIFAR10("cifar10", train=True, download=True, transform=T.ToTensor())

    means = (dataset.data / 255).mean(axis=(0, 1, 2))
    stds = (dataset.data / 255).std(axis=(0, 1, 2))

    train_transforms = T.Compose(
        [
            T.RandomResizedCrop(size=32, scale=(0.8, 1.1)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomAdjustSharpness(sharpness_factor=2),
            T.ToTensor(),
            T.Normalize(mean=means, std=stds),
        ]
    )

    test_transforms = T.Compose([T.ToTensor(), T.Normalize(mean=means, std=stds)])

    dataset_train = CIFAR10(
        "cifar10", train=True, download=True, transform=train_transforms
    )
    dataset_valid = CIFAR10(
        "cifar10", train=False, download=True, transform=test_transforms
    )

    train_loader = DataLoader(dataset_train, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
    valid_loader = DataLoader(dataset_valid, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = func().to(device)

    optimizer = Adam(model.parameters(), lr=1e-3)

    loss_fn = nn.CrossEntropyLoss()

    whole_train_valid_cycle(
        model, train_loader, valid_loader, optimizer, loss_fn, device, threshold, title
    )

    torch.save(predict(model, valid_loader, device), "prediction.pt")


In [ ]:
import torch.nn as nn


def create_simple_conv_cifar():
    return nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
        nn.ReLU(),

        nn.MaxPool2d(2),

        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),

        nn.MaxPool2d(2),

        nn.Flatten(),

        nn.Linear(8 * 8 * 32, 1024),
        nn.ReLU(),
        nn.Linear(1024, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )


In [ ]:
main(create_simple_conv_cifar, 0.7, 'create_simple_conv_cifar')

## Задание 9

Напишите функцию `predict_tta`. Она должна принимать на вход нейронную сеть, даталоадер, `torch.device` и количество итераций по даталоадеру. Она должна иметь следующую сигнатуру: `def predict_tta(model: nn.Module, loader: DataLoader, device: torch.device, iterations: int = 2):`.

В этой функции мы применим технику, которую кратко обсуждали на занятии - Test Time Augmentation. Основная идея заключается в том, чтобы сделать для каждой картинки из тестовой выборки несколько аугментированных вариантов и сделать для них предсказания. Потом эти предсказания усреднить и использовать как обычно. В синтаксисе `PyTorch` это вырождается в создание тестового датасета со случайными аугментациями (либо как на обучающей выборке, либо чуть более слабыми). После этого нужно проитерироваться по созданному датасету несколько раз и усреднить ответы модели.

Внутри функции сделайте следующие шаги:

 1. Запустите цикл по количеству итераций
 2. Внутри цикла проитерируйтесь по даталоадеру
 3. Запишите ответы (не классы, а сырые выходы нейросети) модели в один большой тензор размера [N, C], где C - число классов, а N - количество объектов в датасете (то есть мы должны иметь для каждого объекта вектор из выходов нейросети, логиты)
 4. Сделайте из этих тензоров один огромный тензор размера [N, C, iterations], усредните его по итерациям, чтобы его размер стал [N, C]
 5. Дальше предскажите классы для объектов по этому тензору как аргмакс, верните их из функции

Ваша функция должна возвращать тензор с **классами**. Не забудьте переводить модель в режим применения и навешивать декторатор для выключения подсчета градиентов.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


@torch.inference_mode()
def predict_tta(model: nn.Module, loader: DataLoader, device: torch.device, iterations: int = 2):
    model.eval()

    prediction = []

    for iteration in range(iterations):
        single_prediction = []

        for x, _ in loader:
            output = model(x.to(device)).cpu()

            single_prediction.append(output)

        prediction.append(torch.vstack(single_prediction))

    prediction = torch.argmax(torch.mean(torch.stack(prediction), dim=0), dim=1)

    return prediction


## Задание 10

Обучите любую модель на датасете CIFAR10, добейтесь значения метрики Accuracy в 90% на тестовой выборке. Вам нужно сдать код с функцией, которая возвращает модель, назовите эту функцию `create_advanced_conv_cifar`. Она не принимает аргументов и возвращает модель. Также сдайте предсказание для тестовой выборки датасета CIFAR10, воспользуйтесь функцией `predict_tta` из предыдущего задания (сравните результаты с помощью обычного `predict` и `predict_tta`, оцените какой прирост качества дает последняя), для того, чтобы использовать `predict_tta` нужно создать отдельно тестовый даталоадер с аугментациями. Воспользуйтесь `torch.save` для записи тензора с результатом предсказания на диск.


In [ ]:
from torch.optim.lr_scheduler import StepLR


def whole_train_valid_cycle(
    model, train_loader, valid_loader, optimizer, loss_fn, scheduler, device, threshold, title
):
    train_loss_history, valid_loss_history = [], []
    train_accuracy_history, valid_accuracy_history = [], []

    for epoch in range(100):
        train_loss, train_accuracy = train(
            model, train_loader, optimizer, loss_fn, device
        )
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn, device)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        train_accuracy_history.append(train_accuracy)
        valid_accuracy_history.append(valid_accuracy)

        clear_output(wait=True)

        plot_stats(
            train_loss_history,
            valid_loss_history,
            train_accuracy_history,
            valid_accuracy_history,
            title,
        )

        scheduler.step()

        if valid_accuracy >= threshold:
            break


def main(func, threshold, title):
    set_seed(0xDEADF00D)

    dataset = CIFAR10("cifar10", train=True, download=True, transform=T.ToTensor())

    means = (dataset.data / 255).mean(axis=(0, 1, 2))
    stds = (dataset.data / 255).std(axis=(0, 1, 2))

    train_transforms = T.Compose(
        [
            T.RandomResizedCrop(size=32, scale=(0.8, 1.1)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomAdjustSharpness(sharpness_factor=2),
            T.ToTensor(),
            T.Normalize(mean=means, std=stds),
        ]
    )

    test_transforms = T.Compose([T.ToTensor(), T.Normalize(mean=means, std=stds)])

    dataset_train = CIFAR10(
        "cifar10", train=True, download=True, transform=train_transforms
    )
    dataset_valid = CIFAR10(
        "cifar10", train=False, download=True, transform=test_transforms
    )

    train_loader = DataLoader(dataset_train, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
    valid_loader = DataLoader(dataset_valid, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = func().to(device)

    optimizer = Adam(model.parameters(), lr=1e-3)
    scheduler = StepLR(optimizer, step_size=20, gamma=0.1)

    loss_fn = nn.CrossEntropyLoss()

    whole_train_valid_cycle(
        model, train_loader, valid_loader, optimizer, loss_fn, scheduler, device, threshold, title
    )

    torch.save(model.state_dict(), "model.pt")

    dataset_valid_tta = CIFAR10("cifar10", train=False, download=True, transform=train_transforms)
    valid_tta_loader = DataLoader(dataset_valid_tta, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

    torch.save(predict_tta(model, valid_tta_loader, device, iterations=20), "prediction.pt")


In [ ]:
# файл для сдачи
import torch.nn as nn


def create_advanced_conv_cifar():
    return nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.MaxPool2d(2),
        nn.Dropout2d(p=0.2),

        nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),

        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),

        nn.MaxPool2d(2),
        nn.Dropout2d(p=0.2),

        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),

        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),

        nn.MaxPool2d(2),
        nn.Dropout2d(p=0.2),

        nn.Flatten(),

        nn.Linear(4 * 4 * 128, 512),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.3),
        nn.ReLU(),
        nn.Linear(512, 100)
    )


In [ ]:
main(create_advanced_conv_cifar, 0.9, 'create_advanced_conv_cifar')